# Ροή εργασίας με ανθρώπινη παρέμβαση στο Microsoft Agent Framework

## 🎯 Στόχοι μάθησης

Σε αυτό το σημειωματάριο, θα μάθετε πώς να υλοποιείτε ροές εργασίας **με ανθρώπινη παρέμβαση** χρησιμοποιώντας το `RequestInfoExecutor` του Microsoft Agent Framework. Αυτό το ισχυρό μοτίβο επιτρέπει την παύση ροών εργασίας AI για τη συλλογή ανθρώπινης εισροής, καθιστώντας τους πράκτορές σας διαδραστικούς και δίνοντας στους ανθρώπους τον έλεγχο κρίσιμων αποφάσεων.

## 🔄 Τι είναι η ανθρώπινη παρέμβαση;

Η **ανθρώπινη παρέμβαση (HITL)** είναι ένα σχεδιαστικό μοτίβο όπου οι πράκτορες AI σταματούν την εκτέλεση για να ζητήσουν ανθρώπινη εισροή πριν συνεχίσουν. Αυτό είναι απαραίτητο για:

- ✅ **Κρίσιμες αποφάσεις** - Λάβετε ανθρώπινη έγκριση πριν από σημαντικές ενέργειες
- ✅ **Αμφίβολες καταστάσεις** - Αφήστε τους ανθρώπους να διευκρινίσουν όταν η AI είναι αβέβαιη
- ✅ **Προτιμήσεις χρηστών** - Ζητήστε από τους χρήστες να επιλέξουν μεταξύ πολλαπλών επιλογών
- ✅ **Συμμόρφωση & ασφάλεια** - Διασφαλίστε ανθρώπινη εποπτεία για ρυθμιζόμενες λειτουργίες
- ✅ **Διαδραστικές εμπειρίες** - Δημιουργήστε συνομιλιακούς πράκτορες που ανταποκρίνονται σε εισροές χρηστών

## 🏗️ Πώς λειτουργεί στο Microsoft Agent Framework

Το πλαίσιο παρέχει τρία βασικά στοιχεία για τη HITL:

1. **`RequestInfoExecutor`** - Ένας ειδικός εκτελεστής που παύει τη ροή εργασίας και εκπέμπει ένα `RequestInfoEvent`
2. **`RequestInfoMessage`** - Βασική κλάση για τυποποιημένα φορτία αιτήσεων που αποστέλλονται σε ανθρώπους
3. **`RequestResponse`** - Συνδέει τις ανθρώπινες απαντήσεις με τις αρχικές αιτήσεις μέσω `request_id`

**Μοτίβο ροής εργασίας:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Παράδειγμά μας: Κράτηση ξενοδοχείου με ανθρώπινη επιβεβαίωση

Θα βασιστούμε στη ροή εργασίας με συνθήκη προσθέτοντας ανθρώπινη επιβεβαίωση **πριν** προταθούν εναλλακτικοί προορισμοί:

1. Ο χρήστης ζητά προορισμό (π.χ., "Παρίσι")
2. Ο `availability_agent` ελέγχει αν υπάρχουν διαθέσιμα δωμάτια
3. **Αν δεν υπάρχουν δωμάτια** → ο `confirmation_agent` ρωτά "Θα θέλατε να δείτε εναλλακτικές;"
4. Η ροή εργασίας **παύει** χρησιμοποιώντας το `RequestInfoExecutor`
5. Ο **άνθρωπος απαντά** "ναι" ή "όχι" μέσω της κονσόλας
6. Ο `decision_manager` κατευθύνει ανάλογα με την απάντηση:
   - **Ναι** → Εμφάνιση εναλλακτικών προορισμών
   - **Όχι** → Ακύρωση της αίτησης κράτησης
7. Εμφάνιση τελικού αποτελέσματος

Αυτό δείχνει πώς να δίνετε στους χρήστες τον έλεγχο στις προτάσεις του πράκτορα!

---

Ας ξεκινήσουμε! 🚀


## Βήμα 1: Εισαγωγή Απαραίτητων Βιβλιοθηκών

Εισάγουμε τα τυπικά στοιχεία του Agent Framework συν τις **κλάσεις ειδικές για τον ανθρώπινο κρίκο στη ροή εργασίας**:
- `RequestInfoExecutor` - Εκτελεστής που διακόπτει τη ροή εργασίας για ανθρώπινη εισαγωγή
- `RequestInfoEvent` - Γεγονός που εκπέμπεται όταν ζητείται ανθρώπινη εισαγωγή
- `RequestInfoMessage` - Βασική κλάση για τυποποιημένα αιτήματα
- `RequestResponse` - Συνδέει τις ανθρώπινες απαντήσεις με τα αιτήματα
- `WorkflowOutputEvent` - Γεγονός για τον εντοπισμό εξόδων ροής εργασίας


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Βήμα 2: Ορισμός Μοντέλων Pydantic για Δομημένα Αποτελέσματα

Αυτά τα μοντέλα ορίζουν το **σχήμα** που θα επιστρέφουν οι πράκτορες. Διατηρούμε όλα τα μοντέλα από τη ροή εργασίας υπό προϋποθέσεις και προσθέτουμε:

**Νέο για το Ανθρώπινο Στοιχείο στον Κύκλο:**
- `HumanFeedbackRequest` - Υποκλάση του `RequestInfoMessage` που ορίζει το φορτίο αιτήματος που αποστέλλεται σε ανθρώπους
  - Περιέχει το `prompt` (ερώτηση προς τον άνθρωπο) και το `destination` (περίληψη για την μη διαθέσιμη πόλη)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Βήμα 3: Δημιουργία του Εργαλείου Κράτησης Ξενοδοχείου

Το ίδιο εργαλείο από την ροή εργασίας υπό προϋποθέσεις - ελέγχει αν υπάρχουν διαθέσιμα δωμάτια στον προορισμό.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Βήμα 4: Ορισμός Συναρτήσεων Συνθηκών για Δρομολόγηση

Χρειαζόμαστε **τέσσερις συναρτήσεις συνθηκών** για τη ροή εργασίας με συμμετοχή ανθρώπου:

**Από τη ροή εργασίας με συνθήκες:**
1. `has_availability_condition` - Δρομολογεί όταν υπάρχουν διαθέσιμα ξενοδοχεία
2. `no_availability_condition` - Δρομολογεί όταν δεν υπάρχουν διαθέσιμα ξενοδοχεία

**Νέο για τη ροή εργασίας με συμμετοχή ανθρώπου:**
3. `user_wants_alternatives_condition` - Δρομολογεί όταν ο χρήστης λέει "ναι" σε εναλλακτικές
4. `user_declines_alternatives_condition` - Δρομολογεί όταν ο χρήστης λέει "όχι" σε εναλλακτικές


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Βήμα 5: Δημιουργία του Εκτελεστή Διαχειριστή Αποφάσεων

Αυτή είναι η **καρδιά του προτύπου ανθρώπου-στον-βρόχο**! Ο `DecisionManager` είναι ένας προσαρμοσμένος `Executor` που:

1. **Λαμβάνει ανθρώπινη ανατροφοδότηση** μέσω αντικειμένων `RequestResponse`
2. **Επεξεργάζεται την απόφαση του χρήστη** (ναι/όχι)
3. **Κατευθύνει τη ροή εργασίας** στέλνοντας μηνύματα στους κατάλληλους πράκτορες

Κύρια χαρακτηριστικά:
- Χρησιμοποιεί τον διακοσμητή `@handler` για να εκθέσει μεθόδους ως βήματα της ροής εργασίας
- Λαμβάνει `RequestResponse[HumanFeedbackRequest, str]` που περιέχει τόσο το αρχικό αίτημα όσο και την απάντηση του χρήστη
- Παράγει απλά μηνύματα "ναι" ή "όχι" που ενεργοποιούν τις συναρτήσεις συνθηκών μας


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Βήμα 6: Δημιουργήστε Προσαρμοσμένο Εκτελεστή Εμφάνισης

Ο ίδιος εκτελεστής εμφάνισης από την υπό όρους ροή εργασίας - παράγει τα τελικά αποτελέσματα ως έξοδο ροής εργασίας.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Βήμα 7: Φόρτωση Μεταβλητών Περιβάλλοντος

Διαμορφώστε τον πελάτη LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Βήμα 8: Δημιουργία Πρακτόρων AI και Εκτελεστών

Δημιουργούμε **έξι συνιστώσες ροής εργασίας**:

**Πράκτορες (περιτυλιγμένοι σε AgentExecutor):**
1. **availability_agent** - Ελέγχει τη διαθεσιμότητα ξενοδοχείου χρησιμοποιώντας το εργαλείο
2. **confirmation_agent** - 🆕 Ετοιμάζει το αίτημα ανθρώπινης επιβεβαίωσης
3. **alternative_agent** - Προτείνει εναλλακτικές πόλεις (όταν ο χρήστης λέει ναι)
4. **booking_agent** - Ενθαρρύνει την κράτηση (όταν υπάρχουν διαθέσιμα δωμάτια)
5. **cancellation_agent** - 🆕 Διαχειρίζεται μήνυμα ακύρωσης (όταν ο χρήστης λέει όχι)

**Ειδικοί Εκτελεστές:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` που παύει τη ροή εργασίας για ανθρώπινη εισροή
7. **decision_manager** - 🆕 Προσαρμοσμένος εκτελεστής που δρομολογεί βάσει ανθρώπινης απάντησης (ήδη ορισμένος παραπάνω)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Βήμα 9: Δημιουργία της ροής εργασίας με ανθρώπινη παρέμβαση

Τώρα κατασκευάζουμε το γράφημα της ροής εργασίας με **υπό όρους δρομολόγηση** που περιλαμβάνει τη διαδρομή με ανθρώπινη παρέμβαση:

**Δομή Ροής Εργασίας:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Κύρια Ακμές:**
- `availability_agent → confirmation_agent` (όταν δεν υπάρχουν δωμάτια)
- `confirmation_agent → prepare_human_request` (μετατροπή τύπου)
- `prepare_human_request → request_info_executor` (παύση για τον άνθρωπο)
- `request_info_executor → decision_manager` (πάντα - παρέχει RequestResponse)
- `decision_manager → alternative_agent` (όταν ο χρήστης λέει "ναι")
- `decision_manager → cancellation_agent` (όταν ο χρήστης λέει "όχι")
- `availability_agent → booking_agent` (όταν υπάρχουν διαθέσιμα δωμάτια)
- Όλες οι διαδρομές τελειώνουν στο `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Βήμα 10: Εκτέλεση Περιπτώσεως Δοκιμής 1 - Πόλη ΧΩΡΙΣ Διαθεσιμότητα (Παρίσι με Ανθρώπινη Επιβεβαίωση)

Αυτή η δοκιμή δείχνει τον **πλήρη κύκλο ανθρώπου-στο-βρόχο**:

1. Αίτηση ξενοδοχείου στο Παρίσι
2. ο availability_agent ελέγχει → Χωρίς δωμάτια
3. ο confirmation_agent δημιουργεί ερώτηση κατάλληλη για άνθρωπο
4. ο request_info_executor **παγώνει τη ροή εργασίας** και εκπέμπει `RequestInfoEvent`
5. **Η εφαρμογή ανιχνεύει το συμβάν και ζητά από το χρήστη στην κονσόλα**
6. Ο χρήστης πληκτρολογεί "ναι" ή "όχι"
7. Η εφαρμογή στέλνει την απάντηση μέσω `send_responses_streaming()`
8. ο decision_manager κατευθύνει ανάλογα με την απάντηση
9. Εμφανίζεται το τελικό αποτέλεσμα

**Κύριο Πρότυπο:**
- Χρησιμοποιήστε `workflow.run_stream()` για την πρώτη επανάληψη
- Χρησιμοποιήστε `workflow.send_responses_streaming(pending_responses)` για τις επόμενες επαναλήψεις
- Ακούστε το `RequestInfoEvent` για να ανιχνεύσετε πότε απαιτείται ανθρώπινη εισαγωγή
- Ακούστε το `WorkflowOutputEvent` για να καταγράψετε τα τελικά αποτελέσματα


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Βήμα 11: Εκτέλεση Δοκιμής 2 - Πόλη ΜΕ Διαθεσιμότητα (Στοκχόλμη - Δεν Απαιτείται Ανθρώπινη Εισροή)

Αυτή η δοκιμή δείχνει τη **άμεση διαδρομή** όταν υπάρχουν διαθέσιμα δωμάτια:

1. Αίτημα για ξενοδοχείο στη Στοκχόλμη
2. ο availability_agent ελέγχει → Διαθέσιμα δωμάτια ✅
3. ο booking_agent προτείνει κράτηση
4. το display_result δείχνει επιβεβαίωση
5. **Δεν απαιτείται ανθρώπινη εισροή!**

Η ροή εργασίας παρακάμπτει εντελώς τον ανθρώπινο παράγοντα όταν υπάρχουν διαθέσιμα δωμάτια.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Κύρια Συμπεράσματα και Καλύτερες Πρακτικές Ανθρώπου-στο-Βρόχο

### ✅ Τι Έχετε Μάθει:

#### 1. **Πρότυπο RequestInfoExecutor**
Το πρότυπο ανθρώπου-στο-βρόχο στο Microsoft Agent Framework χρησιμοποιεί τρία βασικά συστατικά:
- `RequestInfoExecutor` - Παύει τη ροή εργασίας και εκπέμπει γεγονότα
- `RequestInfoMessage` - Βασική κλάση για τυποποιημένα αιτήματα (κάντε υποκλάση!)
- `RequestResponse` - Συσχετίζει τις απαντήσεις ανθρώπων με τα αρχικά αιτήματα

**Σημαντική Κατανόηση:**
- Το `RequestInfoExecutor` ΔΕΝ συλλέγει μόνο του τις εισροές - απλά διακόπτει τη ροή εργασίας
- Ο κώδικας της εφαρμογής σας πρέπει να ακούει το `RequestInfoEvent` και να συλλέγει εισροές
- Πρέπει να καλέσετε το `send_responses_streaming()` με dict που συσχετίζει `request_id` με την απάντηση του χρήστη

#### 2. **Πρότυπο Ροής Εκτέλεσης (Streaming Execution)**
```python
# Πρώτη επανάληψη
stream = workflow.run_stream(initial_request)

# Επόμενες επαναλήψεις (μετά την εισαγωγή από τον άνθρωπο)
stream = workflow.send_responses_streaming(pending_responses)

# Επεξεργάζεστε πάντα τα γεγονότα
events = [event async for event in stream]
```

#### 3. **Αρχιτεκτονική με Οδηγίες από Γεγονότα**
Ακούστε για συγκεκριμένα γεγονότα για να ελέγξετε τη ροή εργασίας:
- `RequestInfoEvent` - Απαραίτητη ανθρώπινη εισροή (η ροή εργασίας διακόπτεται)
- `WorkflowOutputEvent` - Διαθέσιμο το τελικό αποτέλεσμα (η ροή εργασίας ολοκληρώθηκε)
- `WorkflowStatusEvent` - Αλλαγές κατάστασης (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, κτλ)

#### 4. **Προσαρμοσμένοι Εκτελεστές με @handler**
Ο `DecisionManager` δείχνει πώς να δημιουργείτε εκτελεστές που:
- Χρησιμοποιούν το διακοσμητή `@handler` για να εκθέτουν μεθόδους ως βήματα ροής εργασίας
- Λαμβάνουν τύπους μηνυμάτων (π.χ. `RequestResponse[HumanFeedbackRequest, str]`)
- Προωθούν τη ροή εργασίας στέλνοντας μηνύματα σε άλλους εκτελεστές
- Έχουν πρόσβαση στο περιβάλλον μέσω `WorkflowContext`

#### 5. **Υπό Όρους Δρομολόγηση με Ανθρώπινες Αποφάσεις**
Μπορείτε να δημιουργήσετε συναρτήσεις όρων που αξιολογούν τις ανθρώπινες απαντήσεις:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Πραγματικές Εφαρμογές:

1. **Ροές Έγκρισης**
   - Λάβετε τη θεώρηση του διευθυντή πριν την επεξεργασία των αναφορών εξόδων
   - Απαιτήστε ανθρώπινη αναθεώρηση πριν την αποστολή αυτοματοποιημένων email
   - Επιβεβαιώστε συναλλαγές υψηλής αξίας πριν την εκτέλεση

2. **Μέτρηση Περιεχομένου**
   - Σημάνετε αμφίβολα περιεχόμενα για αναθεώρηση ανθρώπων
   - Ζητήστε από τους επιμελητές να πάρουν τελική απόφαση σε ακραίες περιπτώσεις
   - Ανέβασε σε ανθρώπους όταν η εμπιστοσύνη της AI είναι χαμηλή

3. **Εξυπηρέτηση Πελατών**
   - Αφήστε την AI να χειριστεί αυτόματα τις ρουτίνες ερωτήσεις
   - Αναθέστε σε ανθρώπινους πράκτορες τα σύνθετα ζητήματα
   - Ρωτήστε τον πελάτη αν θέλει να μιλήσει με άνθρωπο

4. **Επεξεργασία Δεδομένων**
   - Ζητήστε από ανθρώπους να επιλύσουν ασαφή δεδομένα
   - Επιβεβαιώστε τις ερμηνείες AI για ασαφή έγγραφα
   - Αφήστε τους χρήστες να επιλέξουν ανάμεσα σε πολλαπλές έγκυρες ερμηνείες

5. **Συστήματα Κρίσιμης Ασφάλειας**
   - Απαιτήστε ανθρώπινη επιβεβαίωση πριν από μη αναστρέψιμες ενέργειες
   - Λάβετε έγκριση πριν από πρόσβαση σε ευαίσθητα δεδομένα
   - Επιβεβαιώστε αποφάσεις σε ρυθμιζόμενους κλάδους (υγεία, χρηματοοικονομικά)

6. **Διαδραστικοί Πράκτορες**
   - Δημιουργήστε συνομιλητικά bots που κάνουν απορίες παρακολούθησης
   - Φτιάξτε βοηθούς που οδηγούν τους χρήστες μέσω περίπλοκων διαδικασιών
   - Σχεδιάστε πράκτορες που συνεργάζονται βήμα-βήμα με ανθρώπους

### 🔄 Σύγκριση: Με ή Χωρίς Άνθρωπο-στο-Βρόχο

| Χαρακτηριστικό | Υπό Όρους Ροή Εργασίας | Ροή Εργασίας Ανθρώπου-στο-Βρόχο |
|---------|---------------------|---------------------------|
| **Εκτέλεση** | Μοναδική `workflow.run()` | Βρόχος με `run_stream()` + `send_responses_streaming()` |
| **Εισροή Χρήστη** | Καμία (πλήρως αυτοματοποιημένη) | Διαδραστικές ερωτήσεις μέσω `input()` ή UI |
| **Συστατικά** | Πράκτορες + Εκτελεστές | + RequestInfoExecutor + DecisionManager |
| **Γεγονότα** | Μόνο AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, κτλ |
| **Παύση** | Χωρίς παύση | Η ροή εργασίας παύει στο RequestInfoExecutor |
| **Έλεγχος Ανθρώπου** | Χωρίς ανθρώπινο έλεγχο | Οι άνθρωποι παίρνουν βασικές αποφάσεις |
| **Χρήση** | Αυτοματοποίηση | Συνεργασία & επίβλεψη |

### 🚀 Προχωρημένα Πρότυπα:

#### Πολλαπλά Σημεία Απόφασης Ανθρώπου
Μπορείτε να έχετε πολλούς κόμβους `RequestInfoExecutor` στην ίδια ροή εργασίας:
```python
.add_edge(agent1, request_info_1)  # Πρώτη ανθρώπινη απόφαση
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Δεύτερη ανθρώπινη απόφαση
.add_edge(decision_manager_2, final_agent)
```

#### Διαχείριση Χρονικού Ορίου
Υλοποιήστε χρονικά όρια για τις απαντήσεις ανθρώπων:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Προεπιλογή στην ασφαλή επιλογή
```

#### Πλούσια Ενσωμάτωση UI
Αντί για `input()`, ενσωματώστε με web UI, Slack, Teams, κτλ.:
```python
if isinstance(event, RequestInfoEvent):
    # Αποστολή ειδοποίησης στο προτιμώμενο κανάλι του χρήστη
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Υπό Όρους Άνθρωπος-στο-Βρόχο
Ζητήστε ανθρώπινη εισροή μόνο σε συγκεκριμένες περιπτώσεις:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Διαβίβαση σε άνθρωπο μόνο αν η εμπιστοσύνη είναι χαμηλή ή η τιμή είναι υψηλή
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Καλύτερες Πρακτικές:

1. **Πάντα να Κάνετε Υποκλάση το RequestInfoMessage**
   - Παρέχει ασφάλεια τύπων και επαλήθευση
   - Επιτρέπει πλούσιο περιβάλλον για απόδοση UI
   - Διευκρινίζει την πρόθεση κάθε τύπου αιτήματος

2. **Χρησιμοποιήστε Περιγραφικά Προτροπές**
   - Συμπεριλάβετε το πλαίσιο για το τι ζητάτε
   - Εξηγήστε τις συνέπειες κάθε επιλογής
   - Κρατήστε τις ερωτήσεις απλές και σαφείς

3. **Αντιμετωπίστε Απρόβλεπτη Εισροή**
   - Επαληθεύστε τις απαντήσεις χρηστών
   - Παρέχετε προεπιλογές για μη έγκυρη εισροή
   - Δώστε σαφή μηνύματα σφάλματος

4. **Παρακολουθήστε τα Request IDs**
   - Χρησιμοποιήστε τη συσχέτιση ανάμεσα σε request_id και απαντήσεις
   - Μην προσπαθείτε να διαχειριστείτε κατάσταση χειροκίνητα

5. **Σχεδιάστε για Μη-Μπλοκάρισμα**
   - Μην μπλοκάρετε νήματα περιμένοντας εισροή
   - Χρησιμοποιήστε ασύγχρονα πρότυπα παντού
   - Υποστηρίξτε ταυτόχρονες εκδόσεις ροών εργασίας

### 📚 Σχετικές Έννοιες:

- **Μεσολάβηση Πράκτορα** - Παρεμβολή κλήσεων πράκτορα (προηγούμενο σημειωματάριο)
- **Διαχείριση Κατάστασης Ροής Εργασίας** - Διατήρηση κατάστασης ανάμεσα σε εκτελέσεις
- **Πολυ-πρακτορική Συνεργασία** - Συνδυασμός ανθρώπου-στο-βρόχο με ομάδες πρακτόρων
- **Αρχιτεκτονικές Οδηγούμενες από Γεγονότα** - Δημιουργία αντιδραστικών συστημάτων με γεγονότα

---

### 🎓 Συγχαρητήρια!

Έχετε κυριαρχήσει στις ροές εργασίας ανθρώπου-στο-βρόχο με το Microsoft Agent Framework! Τώρα ξέρετε πώς να:
- ✅ Διακόπτετε ροές εργασίας για να συλλέγετε ανθρώπινη εισροή
- ✅ Χρησιμοποιείτε RequestInfoExecutor και RequestInfoMessage
- ✅ Διαχειρίζεστε εκτέλεση ροής με γεγονότα
- ✅ Δημιουργείτε προσαρμοσμένους εκτελεστές με @handler
- ✅ Κατευθύνετε ροές εργασίας βάσει ανθρώπινων αποφάσεων
- ✅ Δημιουργείτε διαδραστικούς πράκτορες AI που συνεργάζονται με ανθρώπους

**Αυτό είναι ένα κρίσιμο πρότυπο για τη δημιουργία αξιόπιστων, ελεγχόμενων συστημάτων AI!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
